# Sprint 4 - Agentic labeling, hierarchy, O*NET alignment, stability, gap analysis and explainability (Gemini)
**Runtime -> Run all.** Code folder: `MyDrive/CAPESTONE PROJECT/SPRINT4_READY/` (or the project root). Put `GEMINI_API_KEY` in `SPRINT4_READY/.env` (fallback: `CAPESTONE PROJECT/.env`). The 80 Leiden leaves and 3,015 memberships are frozen; Gemini labels, groups and judges but never changes task membership. Claude is not used in Sprint 4.

When it finishes, download **`Sprint4_RETURN_TO_REPO.zip`** from `CAPESTONE PROJECT` and extract it into the local `CAPESTONE PROJECT` folder. The run is resumable: re-running reuses every cached Gemini response.

In [ ]:
# 1. Mount Drive, set BASE (project root) and the code folder, extend sys.path
import os, sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/CAPESTONE PROJECT")
else:                                   # local/offline testing only
    BASE = Path(os.environ.get("SPRINT4_BASE", Path.cwd()))
assert BASE.exists(), f"Project folder not found: {BASE}"
CODE = BASE / "SPRINT4_READY" if (BASE / "SPRINT4_READY" / "sprint4_src").exists() else BASE
for p in (CODE, BASE):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
os.chdir(BASE)
print("Project folder:", BASE)
print("Code folder   :", CODE)

In [ ]:
# 2. Install dependencies (only what is missing)
import importlib.util, subprocess
MOCK = os.environ.get("SPRINT4_MOCK") == "1"      # offline test double; never set in Colab
def have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except (ImportError, ValueError):
        return False
need = {"google.genai": "google-genai", "pydantic": "pydantic", "sklearn": "scikit-learn", "scipy": "scipy", "pandas": "pandas", "numpy": "numpy"}
if not MOCK:
    need.update({"sentence_transformers": "sentence-transformers", "sentencepiece": "sentencepiece"})
missing = [pkg for mod, pkg in need.items() if not have(mod)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("dependencies ready" + (f" (installed: {missing})" if missing else ""))

In [ ]:
# 3. Load .env (SPRINT4_READY/.env wins over CAPESTONE PROJECT/.env) and print the SAFE configuration (no key values)
from sprint4_agentic_pipeline import make_settings, preflight, run, validate, show, safe_config
cfg = safe_config(BASE)
print("LLM provider:", cfg["LLM provider"])
print("Gemini key present:", cfg["Gemini key present"])
print("Gemini model:", cfg["Gemini model"])
print("Mock mode:", cfg["Mock mode"])
for line in cfg["env files searched"]:
    print("  .env", line)

In [ ]:
# 4. Gemini preflight: key, google-genai, model setting, frozen/Sentence-T5/O*NET files, writable folders, readable ledger/cache.
#    No generation request is spent on testing the key.
pre = preflight(BASE)
print({k: pre[k] for k in ["status", "model", "normal_call_plan", "hard_call_cap", "real_calls_used_so_far", "real_calls_remaining"]})

In [ ]:
# 5. Complete Sprint 4: evidence -> Gemini leaf labeler -> hierarchy search -> Gemini hierarchy builder -> O*NET top-10 -> Gemini O*NET evaluator ->
#    Gemini critic/convergence -> hierarchy stability (5 seeds) -> O*NET gap analysis -> 80 explainability cards -> human-review template -> archives
summary = run(BASE)

In [ ]:
# 6. Validate all outputs and both archives
result = validate(BASE)
for name, c in result["checks"].items():
    print(("PASS " if c["ok"] else "FAIL ") + name)
print("VALIDATION:", result["status"])
assert result["status"] == "PASS", "validation failed - see sprint4_results/validation.json"

In [ ]:
# 7. Final results
import pandas as pd
from IPython.display import display, Markdown
show(BASE)
R = BASE / "sprint4_results"
display(pd.read_csv(R / "leaf_topics_labeled.csv")[["topic_id", "topic_name", "topic_size", "level2_name", "level1_name", "onet_alignment", "final_status"]].head(15))
display(pd.read_csv(R / "onet_gap_analysis.csv").coverage_class.value_counts().rename("leaf topics").to_frame())
display(pd.read_csv(R / "hierarchy_stability.csv")[["topic_id", "topic_name", "level1_parent_persistence", "level2_parent_persistence"]].head(10))
display(Markdown("Report: `sprint4_results/sprint4_evaluation_report.md`"))
print("DOWNLOAD (extract into local CAPESTONE PROJECT):", BASE / "Sprint4_RETURN_TO_REPO.zip")